# Step 2 — Handoff Experiments Notebook

Use this once Step 1 stabilizes. It compares multiple metrics JSON artifacts, applies the same composite uplift rule, and maintains a ranked experiment board with a final `run_notes` column.

## ML rationale for Step 2
Step 2 is a **promotion gate** notebook: it compares candidate experiment outputs against the same ranking-quality criteria used in Step 1.

### Why this helps
- Ensures we do not promote a model/config based only on raw score fit.
- Keeps focus on ranking behavior at decision depth $k$ (what users actually see first).
- Tracks improvement stability over time with a persistent experiment board.

### Core decision rule
For each candidate metrics artifact:
$$
\text{composite\_uplift} = \Delta\text{topk\_target} + \Delta\text{ndcg@k}
$$
Then rank all candidates by composite uplift and keep notes for traceability.

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 200)

## Code logic walkthrough
1. Discover metrics JSON artifacts (glob pattern).
2. Parse each artifact into one normalized row (`delta_topk_target`, `delta_ndcg_at_k`, `composite_uplift`).
3. Rank current batch (`winner_rank`).
4. Append to persistent experiment board and recompute all-time rank (`winner_rank_all_time`).
5. Keep `run_notes` as the final column for human context (listening checks, data drift notes, release decisions).

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RUNS_ROOT = PROJECT_ROOT / 'training' / 'models' / 'fast_tune_weight_sweeps'
BOARD_CSV = RUNS_ROOT / 'step2_experiment_board.csv'

# You can point this glob to broader candidate folders as needed
METRIC_FILES = sorted(RUNS_ROOT.glob('eval_metrics_*.json'))
RUN_NOTE = 'step2 comparison pass'

print('Metrics files found:', len(METRIC_FILES))
for p in METRIC_FILES[:10]:
    print(' -', p.name)

In [ ]:
def row_from_metrics(path: Path) -> dict:
    m = json.loads(path.read_text(encoding='utf-8'))
    w = m.get('composite_eval', {}).get('weights_requested', {})
    d = m.get('model_a', {}).get('composite_vs_raw', {})
    d_topk = float(d.get('delta_topk_target', 0.0))
    d_ndcg = float(d.get('delta_ndcg_at_k', 0.0))
    return {
        'experiment_file': str(path),
        'model_score': float(w.get('model_score', 0.0)),
        'periodicity': float(w.get('periodicity', 0.0)),
        'drum_alignment': float(w.get('drum_alignment', 0.0)),
        'delta_topk_target': d_topk,
        'delta_ndcg_at_k': d_ndcg,
        'composite_uplift': d_topk + d_ndcg,
    }


rows = [row_from_metrics(p) for p in METRIC_FILES]
step2_df = pd.DataFrame(rows)

if not step2_df.empty:
    step2_df['winner_rank'] = step2_df['composite_uplift'].rank(method='dense', ascending=False).astype(int)
    step2_df = step2_df.sort_values('winner_rank').reset_index(drop=True)
    step2_df['run_timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    step2_df['run_notes'] = RUN_NOTE
    cols = [c for c in step2_df.columns if c != 'run_notes'] + ['run_notes']
    step2_df = step2_df[cols]

display(step2_df.head(20))

In [ ]:
if step2_df.empty:
    print('No metrics files found to rank yet.')
else:
    if BOARD_CSV.exists():
        existing = pd.read_csv(BOARD_CSV)
        merged = pd.concat([existing, step2_df], ignore_index=True)
    else:
        merged = step2_df.copy()

    merged['winner_rank_all_time'] = merged['composite_uplift'].rank(method='dense', ascending=False).astype(int)
    cols = [c for c in merged.columns if c != 'run_notes'] + ['run_notes']
    merged = merged[cols]
    BOARD_CSV.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(BOARD_CSV, index=False)
    print(f'Saved board: {BOARD_CSV}')
    display(merged.sort_values(['winner_rank_all_time', 'run_timestamp']).head(25))

## References and citations
1. **Järvelin, K., & Kekäläinen, J. (2002).** Cumulated gain-based evaluation of IR techniques. *ACM TOIS*. https://doi.org/10.1145/582415.582418
2. **Manning, Raghavan, Schütze (2008).** *Introduction to Information Retrieval* (evaluation chapter). https://nlp.stanford.edu/IR-book/
3. **Liu, T.-Y. (2009).** Learning to Rank for Information Retrieval. https://www.nowpublishers.com/article/Details/INR-016
4. **TREC resources** for ranking evaluation practice. https://trec.nist.gov/
5. **Master Brain Bridge API docs** (query contract used to fetch project-indexed evidence). http://127.0.0.1:8787/docs

### Master Brain source pull
If your API key is configured, run the Step 1 Master Brain citation cell and reuse the resulting `master_brain_sources_runtime.json` to enrich this board with project-specific context.